In [1]:
import requests
import pandas as pd

# Fonte: INE — Rendimento médio disponível por habitante (varcd=0009762)
#https://www.ine.pt/xportal/xmain?xpid=INE&xpgid=ine_indicadores&indOcorrCod=0009762&contexto=bd&selTab=tab2


anos = range(2015, 2022)
rows = []

for ano in anos:
    url = (
        f"https://www.ine.pt/ine/json_indicador/pindica.jsp"
        f"?op=2&varcd=0009762&lang=PT&Dim1=S7A{ano}"
    )
    data = requests.get(url).json()

    for entry in data:
        # Cada entry tem uma chave "Dados" com os registos desse ano
        for ano_key, registos in entry.get("Dados", {}).items():
            for r in registos:
                geocod = r.get("geocod", "")
                # Excluir agregados (Portugal, regiões) — só municípios têm geocod numérico
                if not geocod or not geocod[0].isdigit():
                    continue
                rows.append({
                    "ANO":                    int(ano_key),
                    "Município":              r.get("geodsg"),
                    "Household Income (EUR)": pd.to_numeric(r.get("valor"), errors="coerce"),
                })

# Construir o DataFrame final, remover duplicados e ordenar
df_final = (
    pd.DataFrame(rows)
    .dropna(subset=["Household Income (EUR)"])
    .drop_duplicates()
    .sort_values(["ANO", "Município"])
    .reset_index(drop=True)
)

print(f"{len(df_final)} registos | {df_final['Município'].nunique()} municípios | Anos: {sorted(df_final['ANO'].unique())}")
df_final


2359 registos | 335 municípios | Anos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


,ANO,Município,Household Income (EUR)
0,2015,Abrantes,15108
1,2015,Aguiar da Beira,11334
2,2015,Alandroal,11515
3,2015,Albergaria-a-Velha,14776
4,2015,Albufeira,13363
...,...,...,...
2354,2021,Área Metropolitana de Lisboa,23321
2355,2021,Área Metropolitana do Porto,20110
2356,2021,Évora,22087
2357,2021,Ílhavo,20196


In [2]:
#  Média do rendimento por ano (anos com maior rendimento médio) 
df_final.groupby("ANO")["Household Income (EUR)"].mean().round(2).sort_values(ascending=False).reset_index()


,ANO,Household Income (EUR)
0,2021,17253.19
1,2020,16533.68
2,2019,16317.03
3,2018,15741.34
4,2017,15102.47
5,2016,14683.76
6,2015,14164.22


In [3]:
#  Top 10 registos com maior rendimento (ANO + Município, overall) 
df_final.nlargest(10, "Household Income (EUR)").reset_index(drop=True)


,ANO,Município,Household Income (EUR)
0,2021,Oeiras,29727
1,2021,Lisboa,29082
2,2020,Oeiras,28736
3,2019,Oeiras,28718
4,2019,Lisboa,28372
5,2018,Oeiras,28160
6,2020,Lisboa,28072
7,2018,Lisboa,27777
8,2017,Oeiras,27263
9,2021,Cascais,26785


In [4]:
#  Todos os registos ordenados por rendimento decrescente 
df_final.sort_values("Household Income (EUR)", ascending=False).reset_index(drop=True)


,ANO,Município,Household Income (EUR)
0,2021,Oeiras,29727
1,2021,Lisboa,29082
2,2020,Oeiras,28736
3,2019,Oeiras,28718
4,2019,Lisboa,28372
...,...,...,...
2354,2015,Boticas,10547
2355,2015,Celorico de Basto,10335
2356,2016,Cinfães,10234
2357,2015,Resende,10183


In [ ]:
df_final.to_parquet("household-income-15-21.parquet")

: 